# Imports

In [1]:
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import StandardScaler
from scipy.io import loadmat
import matplotlib.pyplot as plt 
import numpy as np 
import os 
import pandas as pd
from scipy.stats import mode  # For consensus calculation

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
from scipy.signal import butter, lfilter
import numpy as np




# Data labeling

In [2]:
columns = [
    'ED_COUNTER',    'ED_INTERPOLATED',    'ED_RAW_CQ',    'ED_AF3',    'ED_F7',
    'ED_F3',    'ED_FC5',    'ED_T7',    'ED_P7',    'ED_O1',
    'ED_O2',    'ED_P8',    'ED_T8',    'ED_FC6',    'ED_F4',
    'ED_F8',    'ED_AF4',    'ED_GYROX',    'ED_GYROY',    'ED_TIMESTAMP',
    'ED_ES_TIMESTAMP',    'ED_FUNC_ID',    'ED_FUNC_VALUE',    'ED_MARKER',    'ED_SYNC_SIGNAL'
]

In [3]:
FOCUSED_ID = 0
UNFOCUSED_ID = 1
DROWSY_ID = 2

def get_state(timestamp):
    if timestamp <= 10*128*60:
        return FOCUSED_ID
    elif timestamp > 20*128*60:
        return UNFOCUSED_ID
    else:
        return DROWSY_ID

# Data preprocessing

In [4]:
# Bandpass filter function
def bandpass_filter(data, low_freq, high_freq, fs, order=4):
    nyquist = 0.5 * fs
    low = low_freq / nyquist
    high = high_freq / nyquist
    b, a = butter(order, [low, high], btype='band')
    return lfilter(b, a, data)

# Brainwave frequency ranges
brainwave_ranges = {
    "Delta": (0.5, 4),
    "Theta": (4, 8),
    "Alpha": (8, 13),
    "Beta": (13, 30)
}

# Parameters
features = []
delta_features = []
theta_features = []
alpha_features = []
beta_features = []
labels = []
SAMPLE_LENGTH_SECOND = 4
FREQUENCY_HZ = 128
SAMPLE_LENGTH_HZ = FREQUENCY_HZ * SAMPLE_LENGTH_SECOND
scaler = StandardScaler(with_mean=True, with_std=True)

# File extraction loop
for i in [3,4,5,6,7, 10,11,12,13,14,17,18,19,20,21,24,25,26,27,31,32,33,34]: 
    print(f"Extracting file {i}")
    mat_data = loadmat(f'/kaggle/input/eeg-data-for-mental-attention-state-detection/EEG Data/eeg_record{i}.mat')
    data = mat_data['o'][0][0]['data']
    eeg_df = pd.DataFrame(data, columns=columns)
    eeg_df.reset_index(inplace=True)
    eeg_df.rename(columns={'index': 'timestamp'}, inplace=True)
    eeg_df['state'] = eeg_df['timestamp'].apply(get_state)
    # df_selected = eeg_df[['ED_AF3', 'ED_AF4', 'ED_F3', 'ED_F4', 'state']]

    # Extract original EEG features
    feature = eeg_df.iloc[:, 4:18].values  # Columns 4 to 17 (0-indexed)
    label = eeg_df['state'].values

    # Scale the original EEG features
    feature = scaler.fit_transform(feature)

    # Apply bandpass filters for each brainwave type
    brainwave_features = {}
    for wave, (low, high) in brainwave_ranges.items():
        filtered = np.apply_along_axis(
            bandpass_filter, 0, feature, low, high, FREQUENCY_HZ
        )
        brainwave_features[wave] = filtered  # Do not scale these

    # Reshape for sample segments
    num_samples = len(feature) // SAMPLE_LENGTH_HZ
    feature = feature[:num_samples * SAMPLE_LENGTH_HZ]
    label = label[:num_samples * SAMPLE_LENGTH_HZ]

    # Original feature
    feature = feature.reshape(num_samples, SAMPLE_LENGTH_HZ, 14, 1)
    label = label.reshape(num_samples, SAMPLE_LENGTH_HZ)
    consensus_labels = mode(label, axis=1)[0].flatten()

    # Append original and brainwave-specific features
    features.append(feature)
    labels.append(consensus_labels)

    for wave in brainwave_ranges.keys():
        brainwave_feature = brainwave_features[wave][:num_samples * SAMPLE_LENGTH_HZ]
        brainwave_feature = brainwave_feature.reshape(num_samples, SAMPLE_LENGTH_HZ, 14, 1)
        if wave == "Delta":
            delta_features.append(brainwave_feature)
        elif wave == "Theta":
            theta_features.append(brainwave_feature)
        elif wave == "Alpha":
            alpha_features.append(brainwave_feature)
        elif wave == "Beta":
            beta_features.append(brainwave_feature)

# Combine all features and labels
features = np.vstack(features)
delta_features = np.vstack(delta_features)
theta_features = np.vstack(theta_features)
alpha_features = np.vstack(alpha_features)
beta_features = np.vstack(beta_features)
labels = np.concatenate(labels)

# Print final shapes
print(f"Delta Features Shape: {delta_features.shape}")
print(f"Theta Features Shape: {theta_features.shape}")
print(f"Alpha Features Shape: {alpha_features.shape}")
print(f"Beta Features Shape: {beta_features.shape}")
print(f"Final Labels Shape: {labels.shape}")


Extracting file 3
Extracting file 4
Extracting file 5
Extracting file 6
Extracting file 7
Extracting file 10
Extracting file 11
Extracting file 12
Extracting file 13
Extracting file 14
Extracting file 17
Extracting file 18
Extracting file 19
Extracting file 20
Extracting file 21
Extracting file 24
Extracting file 25
Extracting file 26
Extracting file 27
Extracting file 31
Extracting file 32
Extracting file 33
Extracting file 34
Delta Features Shape: (17150, 512, 14, 1)
Theta Features Shape: (17150, 512, 14, 1)
Alpha Features Shape: (17150, 512, 14, 1)
Beta Features Shape: (17150, 512, 14, 1)
Final Labels Shape: (17150,)


# Data classes distribution

In [5]:
import numpy as np
unique_classes, class_counts = np.unique(labels, return_counts=True)

# Display the counts
for cls, count in zip(unique_classes, class_counts):
    print(f"Class {cls}: {count} samples")


Class 0: 3450 samples
Class 1: 10250 samples
Class 2: 3450 samples


# Model

In [6]:
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix



In [183]:
from sklearn.utils import shuffle

# Prepare the data
X = features  
y = labels 
# Split the data into three classes
focused_indices = np.where(y == 0)[0]
unfocused_indices = np.where(y == 1)[0]
drowsy_indices = np.where(y == 2)[0]

# Find the number of samples in the 'drowsy' class
num_drowsy_samples = len(drowsy_indices)

# Undersample the unfocused class to match the number of drowsy samples
unfocused_indices_undersampled = np.random.choice(unfocused_indices, num_drowsy_samples, replace=False)

# Combine the indices of all classes
final_indices = np.concatenate([focused_indices, unfocused_indices_undersampled, drowsy_indices])

# Shuffle the indices to mix the samples from all classes
final_indices = shuffle(final_indices, random_state=42)

# Create the balanced dataset using the final indices
X_balanced = X[final_indices]
y_balanced = y[final_indices]

# Print the new class distribution
unique_classes, class_counts = np.unique(y_balanced, return_counts=True)
for cls, count in zip(unique_classes, class_counts):
    print(f"Class {cls}: {count} samples")

Class 0: 3450 samples
Class 1: 3450 samples
Class 2: 3450 samples


In [184]:
X.shape

(17150, 512, 14, 1)

In [185]:
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from mne.decoding import Vectorizer
from sklearn.decomposition import PCA

# X_flattened_balanced = X_balanced.reshape(X_balanced.shape[0], -1) 

# Split the data into training and testing sets
X_train_balanced, X_test_balanced, y_train_balanced, y_test_balanced = train_test_split(X_balanced, y_balanced, test_size=0.2, random_state=42)



In [186]:
print(X_train_balanced.shape,X_test_balanced.shape,  y_train_balanced.shape, y_test_balanced.shape)

(8280, 512, 14, 1) (2070, 512, 14, 1) (8280,) (2070,)


In [187]:
vectorizer = Vectorizer()
Xtrain_feat = vectorizer.fit_transform(X_train_balanced)
Xtest_feat = vectorizer.transform(X_test_balanced)

# PCA for dimensionality reduction
pca = PCA(n_components=0.95, svd_solver = 'full')  
X_train_pca = pca.fit_transform(Xtrain_feat)
X_test_pca = pca.transform(Xtest_feat)

In [188]:
print(X_train_pca.shape,X_test_pca.shape)

(8280, 1539) (2070, 1539)


In [189]:
rlda = LinearDiscriminantAnalysis(solver='eigen', shrinkage=0.90)
rlda.fit(X_train_pca, y_train_balanced)
y_pred_prob = rlda.predict(X_test_pca)

In [190]:
print(classification_report(y_test_balanced, y_pred_prob))

              precision    recall  f1-score   support

           0       0.86      0.74      0.79       685
           1       0.75      0.74      0.75       698
           2       0.59      0.68      0.63       687

    accuracy                           0.72      2070
   macro avg       0.73      0.72      0.73      2070
weighted avg       0.73      0.72      0.73      2070



In [191]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc
import matplotlib.pyplot as plt

In [192]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_pca, y_train_balanced)


RandomForestClassifier(random_state=42)

In [194]:
y_pred = rf.predict(X_test_pca)

In [195]:
print(classification_report(y_test_balanced, y_pred))

              precision    recall  f1-score   support

           0       0.70      0.72      0.71       685
           1       0.63      0.65      0.64       698
           2       0.52      0.49      0.50       687

    accuracy                           0.62      2070
   macro avg       0.61      0.62      0.62      2070
weighted avg       0.61      0.62      0.62      2070



In [206]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Train Lasso
lasso = LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000,  C=0.005,random_state=42)
lasso.fit(X_train_pca, y_train_balanced)

y_pred = lasso.predict(X_test_pca)

# Evaluate the performance
print(classification_report(y_test_balanced, y_pred))

              precision    recall  f1-score   support

           0       0.70      0.86      0.77       685
           1       0.57      0.91      0.70       698
           2       0.59      0.10      0.18       687

    accuracy                           0.63      2070
   macro avg       0.62      0.62      0.55      2070
weighted avg       0.62      0.63      0.55      2070



In [198]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Train L2 Regularized Logistic Regression
ridge = LogisticRegression(penalty='l2', solver='lbfgs', max_iter=1000, random_state=42)
ridge.fit(X_train_pca, y_train_balanced)

# Make predictions on the test set
y_pred = ridge.predict(X_test_pca)

# Evaluate the model's performance
print(classification_report(y_test_balanced, y_pred))


              precision    recall  f1-score   support

           0       0.71      0.69      0.70       685
           1       0.67      0.67      0.67       698
           2       0.51      0.53      0.52       687

    accuracy                           0.63      2070
   macro avg       0.63      0.63      0.63      2070
weighted avg       0.63      0.63      0.63      2070



In [207]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [208]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Train the XGBoost model
xgb_model = xgb.XGBClassifier(
    n_estimators=100,  # Number of trees
    learning_rate=0.1,  # Learning rate
    max_depth=6,  # Maximum depth of the trees
    random_state=42
)
xgb_model.fit(X_train_pca, y_train_balanced)
y_pred = xgb_model.predict(X_test_pca)

# Evaluate the performance
accuracy = accuracy_score(y_test_balanced, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Confusion Matrix
conf_matrix = confusion_matrix(y_test_balanced, y_pred)
print("\nConfusion Matrix:")
print(conf_matrix)

# Classification Report
class_report = classification_report(y_test_balanced, y_pred)
print("\nClassification Report:")
print(class_report)


Accuracy: 0.7401

Confusion Matrix:
[[526  23 136]
 [ 25 517 156]
 [ 85 113 489]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.77      0.80       685
           1       0.79      0.74      0.77       698
           2       0.63      0.71      0.67       687

    accuracy                           0.74      2070
   macro avg       0.75      0.74      0.74      2070
weighted avg       0.75      0.74      0.74      2070



In [209]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Train the XGBoost model
xgb_model = xgb.XGBClassifier(
    n_estimators=200,  # Number of trees
    learning_rate=0.1,  # Learning rate
    max_depth=10,  # Maximum depth of the trees
    random_state=42
)
xgb_model.fit(X_train_pca, y_train_balanced)

y_pred = xgb_model.predict(X_test_pca)

# Evaluate the performance
accuracy = accuracy_score(y_test_balanced, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Confusion Matrix
conf_matrix = confusion_matrix(y_test_balanced, y_pred)
print("\nConfusion Matrix:")
print(conf_matrix)

# Classification Report
class_report = classification_report(y_test_balanced, y_pred)
print("\nClassification Report:")
print(class_report)


Accuracy: 0.7357

Confusion Matrix:
[[527  24 134]
 [ 25 514 159]
 [ 95 110 482]]

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.77      0.79       685
           1       0.79      0.74      0.76       698
           2       0.62      0.70      0.66       687

    accuracy                           0.74      2070
   macro avg       0.74      0.74      0.74      2070
weighted avg       0.74      0.74      0.74      2070



In [210]:
pip install catboost


Note: you may need to restart the kernel to use updated packages.


In [211]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Train the CatBoost model
catboost_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1, 
    depth=6,  
    random_state=42,
    verbose=200  
)


catboost_model.fit(X_train_pca, y_train_balanced)

y_pred = catboost_model.predict(X_test_pca)

# Evaluate the performance
accuracy = accuracy_score(y_test_balanced, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Confusion Matrix
conf_matrix = confusion_matrix(y_test_balanced, y_pred)
print("\nConfusion Matrix:")
print(conf_matrix)

# Classification Report
class_report = classification_report(y_test_balanced, y_pred)
print("\nClassification Report:")
print(class_report)


0:	learn: 1.0595524	total: 958ms	remaining: 15m 56s
200:	learn: 0.4457745	total: 1m 45s	remaining: 6m 58s
400:	learn: 0.2967326	total: 3m 29s	remaining: 5m 12s
600:	learn: 0.2048541	total: 5m 13s	remaining: 3m 28s
800:	learn: 0.1441713	total: 6m 58s	remaining: 1m 43s
999:	learn: 0.1030797	total: 8m 42s	remaining: 0us
Accuracy: 0.7353

Confusion Matrix:
[[534  24 127]
 [ 33 523 142]
 [ 87 135 465]]

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.78      0.80       685
           1       0.77      0.75      0.76       698
           2       0.63      0.68      0.65       687

    accuracy                           0.74      2070
   macro avg       0.74      0.74      0.74      2070
weighted avg       0.74      0.74      0.74      2070

